# Unsloth QLoRA fine-tune on free Colab (T4)

Colab version of [`src/sft_unsloth.py`](https://github.com/Ademo93/unsloth-trl-playground/blob/main/src/sft_unsloth.py): Qwen2.5-3B-Instruct, QLoRA r=16, 200 steps on `yahma/alpaca-cleaned`.

Differences vs the script, forced by the T4 (pre-Ampere, no bf16):
- `fp16=True` instead of `bf16=True`

**Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
%%capture
# Unsloth pins compatible transformers/trl/peft versions itself.
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel  # must be imported before transformers/trl

import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
SEQ_LEN = 1024
MAX_STEPS = 200
SEED = 42

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
def to_text(example):
    prompt = example["instruction"]
    if example.get("input", "").strip():
        prompt += "\n\n" + example["input"]
    return {
        "text": f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>\n"
    }

dataset = load_dataset("yahma/alpaca-cleaned", split="train[:5000]")
dataset = dataset.map(to_text, remove_columns=dataset.column_names)
print(dataset[0]["text"])

In [ ]:
import time

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="outputs/unsloth",
        max_steps=MAX_STEPS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=10,
        max_length=SEQ_LEN,
        fp16=True,  # T4 has no bf16
        seed=SEED,
        report_to="none",
    ),
)

torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
result = trainer.train()
elapsed = time.perf_counter() - start

print(f"s/step: {elapsed / MAX_STEPS:.2f}")
print(f"peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print(f"final loss: {result.training_loss:.4f}")

In [ ]:
# Quick sanity check on the fine-tuned model
FastLanguageModel.for_inference(model)
inputs = tokenizer(
    "<|im_start|>user\nGive three tips to stay productive.<|im_end|>\n<|im_start|>assistant\n",
    return_tensors="pt",
).to(model.device)
output = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Optional: export to GGUF for Ollama

Uncomment to produce a `q4_k_m` GGUF (takes a few extra minutes and disk space):

In [ ]:
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")